# Skin Issue Trainer — Google Colab, public data

Downloads **public, licensed** skin photos automatically, maps their labels onto
our categories, fine-tunes a pretrained Hugging Face vision model, reports
accuracy and F1 every epoch, and exports **ONNX** for the app.

**Nothing to upload.** Runtime → Change runtime type → **T4 GPU**, then
Runtime → **Run all**. The finished model downloads to your computer as a zip.

### The data — and why this dataset

| Source | Licence | Gives us |
|---|---|---|
| **SCIN** — Google's Skin Condition Image Network: 5,033 consumer cases, 10,000+ photos, each labelled by up to three dermatologists, with skin-tone labels | **SCIN Data Use License**: training and sharing allowed, *with attribution*; any attempt to **re-identify** the people pictured is forbidden | acne, redness, dark spots, normal |
| `huynhnhu213/acne` — *optional, off by default* | Apache-2.0 declared, but no source given for the images | extra acne / normal photos |

Every other public "skin disease" upload on Hugging Face was ruled out: no
licence at all (which means all rights reserved), CC BY-NC-ND (no
derivatives — a trained model is one), or copies of the DermNet atlas, whose
images the uploaders had no right to relicense.

### Honest expectations

SCIN is mostly rashes on arms and legs; the slice that matches our categories
is small — roughly **acne ~105, redness ~105, normal ~120, dark spots ~30
images**. That is enough to train a real model and measure it honestly, not
enough for a polished one. **Wrinkles are not included**: no public dataset
with a licence that permits this use labels them.

The optional acne dataset adds thousands of photos but in a different style
from SCIN (close-ups and stock portraits), so the model can learn *which
dataset a photo came from* instead of what is on the skin — the shortcut that
spoiled the local run. It is off by default; section 7 measures the effect if
you turn it on.

> Research artefact, not medical advice. Malignant conditions are removed from
> the data on purpose. Nothing reaches users without the FR-AI-010 gate in
> `ml/README.md`.

## 1 · Setup & installs

In [ ]:
#@title 1a · Install libraries
!pip install -q transformers datasets evaluate accelerate onnx onnxruntime

In [ ]:
#@title 1b · Imports and GPU check
import ast, csv, io, json, math, random, shutil, time, zipfile
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import requests
import torch
import transformers, datasets, evaluate
from datasets import ClassLabel, Dataset, Features, Image as ImageFeature, Value
from PIL import Image
from sklearn.model_selection import train_test_split
from transformers import (AutoImageProcessor, AutoModelForImageClassification,
                          EarlyStoppingCallback, Trainer, TrainingArguments)
from torchvision import transforms

print('torch', torch.__version__, '| transformers', transformers.__version__,
      '| datasets', datasets.__version__)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again.')

## 2 · Configuration

In [ ]:
#@title 2 · Config — the defaults are sensible; edit only if you want to
# Our categories. A category the data cannot supply enough of is dropped in 3d,
# with a message, rather than trained on a handful of photos.
CLASSES = ['acne', 'dark_spots', 'redness', 'normal']

# The backbone. Both pretrained on ImageNet, Apache-2.0; only the head is new.
#   google/vit-base-patch16-224-in21k : 86M params, strongest, ~330 MB ONNX
#   microsoft/resnet-50              : 25M params, ~100 MB ONNX, easier on a phone
MODEL_CHECKPOINT = 'google/vit-base-patch16-224-in21k'  #@param ["google/vit-base-patch16-224-in21k", "microsoft/resnet-50"]

EPOCHS = 15           #@param {type:"integer"}
LEARNING_RATE = 5e-5  #@param {type:"number"}
# Small, because the dataset is small: more updates per epoch.
BATCH_SIZE = 16       #@param {type:"integer"}
VAL_FRACTION = 0.2    #@param {type:"number"}
EARLY_STOPPING_PATIENCE = 4  #@param {type:"integer"}

# --- data --------------------------------------------------------------------
# A SCIN case joins a category only when the dermatologists' combined weight on
# that category's conditions reaches this. 0.5 = they mostly agreed.
MIN_LABEL_WEIGHT = 0.5  #@param {type:"number"}
# Head-and-neck photos only. Closer to selfies, but leaves very little data.
FACE_ONLY = False  #@param {type:"boolean"}
# A category needs at least this many distinct cases (people) to be trained.
MIN_CASES_PER_CLASS = 10  #@param {type:"integer"}
# Optional extra acne/normal photos. Read "Honest expectations" first.
ADD_HF_ACNE_DATASET = False  #@param {type:"boolean"}
HF_ACNE_PER_CLASS = 400  #@param {type:"integer"}
# Photos are downscaled on download: SCIN originals are ~1 MB PNGs, and the
# model sees 224 px anyway.
MAX_IMAGE_SIDE = 512

# --- output ------------------------------------------------------------------
# The model is always zipped and downloaded to your computer. Tick this to also
# keep a copy in Google Drive (it asks for Drive access at the very end).
SAVE_TO_DRIVE = False  #@param {type:"boolean"}

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
RUN_NAME = f"{MODEL_CHECKPOINT.split('/')[-1]}-{time.strftime('%Y%m%d-%H%M')}"
DATA_ROOT = Path('/content/data')
OUTPUT_DIR = Path('/content/skin_issue_model') / RUN_NAME
print('run:', RUN_NAME)

## 3 · Automatic data download

**Why not `load_dataset('google/scin')`?** That copy bundles every image —
12.6 GB — and we need a few hundred of them. Instead this reads SCIN's two
label tables (2 MB) from Google's public bucket, picks the matching cases, and
downloads only their photos.

### How SCIN labels become our categories

Each case lists the conditions its dermatologists considered, weighted by how
confident they were — e.g. `{'Acne': 0.67, 'Folliculitis': 0.33}`. A case's
weights are summed per category below; the highest category wins if it
reaches `MIN_LABEL_WEIGHT`, otherwise the case is left out.

| Our category | SCIN dermatologist labels |
|---|---|
| **acne** | Acne · Comedone · Acneiform eruption |
| **redness** | Rosacea · Perioral Dermatitis · Flushing · Seborrheic Dermatitis *(head and neck only)* |
| **dark_spots** | Post-Inflammatory hyperpigmentation · Melasma · Lentigo · Hyperpigmentation · drug-induced or exogenous pigmentation · Erythema dyschromicum perstans |
| **normal** | Dermatologists recorded *"image quality sufficient, no discernible pathology"* and named no condition |

Deliberately **not** mapped: moles and naevi (not cosmetic "dark spots", and too
close to things only a doctor should judge), *pigmented purpuric eruption* and
*tinea versicolor* (they look discoloured but are something else), and the
contributors' own *"looks healthy"* tag — every one of those cases still
received a dermatologist diagnosis. Any case whose list mentions a **malignant**
condition is dropped entirely.

In [ ]:
#@title 3a · Read SCIN's labels and choose the cases
SCIN = 'https://storage.googleapis.com/dx-scin-public-data/'
SCIN_LICENSE_URL = 'https://huggingface.co/datasets/google/scin'  # licence text in the dataset card

def read_csv(url):
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    return list(csv.DictReader(io.StringIO(response.content.decode('utf-8'))))

cases = {row['case_id']: row for row in read_csv(SCIN + 'dataset/scin_cases.csv')}
labels_by_case = {row['case_id']: row for row in read_csv(SCIN + 'dataset/scin_labels.csv')}
print(f'SCIN: {len(cases)} cases')

SCIN_TO_CATEGORY = {
    'acne': ['Acne', 'Comedone', 'Acneiform eruption'],
    'redness': ['Rosacea', 'Perioral Dermatitis', 'Flushing', 'Seborrheic Dermatitis'],
    'dark_spots': ['Post-Inflammatory hyperpigmentation', 'Melasma', 'Lentigo', 'Hyperpigmentation',
                   'Hyperpigmentation disorder', 'Drug induced abnormal pigmentation of skin',
                   'Melanin pigmentation due to exogenous substance', 'Erythema dyschromicum perstans'],
}
# Seborrheic dermatitis reads as facial redness only on the face and scalp.
HEAD_ONLY = {'Seborrheic Dermatitis'}
MALIGNANT_TERMS = ['melanoma', 'carcinoma', 'scc', 'bcc', 'malignan', 'lymphoma', 'kaposi',
                   'merkel', 'bowen', 'lentigo maligna', 'sarcoma']
NO_PATHOLOGY = 'YES_IMAGE_QUALITY_SUFFICIENT_NO_DISCERNIBLE_PATHOLOGY'
UNGRADABLE = 'NO_IMAGE_QUALITY_INSUFFICIENT'

def skin_tone(case, label):
    # Dermatologist-assessed Fitzpatrick type first, self-reported second.
    raw = label.get('dermatologist_fitzpatrick_skin_type_label_1') or case.get('fitzpatrick_skin_type') or ''
    digits = [ch for ch in raw if ch.isdigit()]
    if not digits:
        return 'unknown'
    return {1: 'I-II', 2: 'I-II', 3: 'III-IV', 4: 'III-IV', 5: 'V-VI', 6: 'V-VI'}.get(int(digits[0]), 'unknown')

def categorise(case, label):
    # Returns (category or None, reason).
    raw = (label.get('weighted_skin_condition_label') or '').strip()
    weights = ast.literal_eval(raw) if raw else {}
    if any(term in name.lower() for name in weights for term in MALIGNANT_TERMS):
        return None, 'malignant condition in the list'
    grades = [label.get(f'dermatologist_gradable_for_skin_condition_{i}', '') for i in (1, 2, 3)]
    on_head = case.get('body_parts_head_or_neck') == 'YES'
    if FACE_ONLY and not on_head:
        return None, 'not head or neck (FACE_ONLY)'
    if not weights:
        if NO_PATHOLOGY in grades:
            return 'normal', ''
        return None, 'ungradable' if UNGRADABLE in grades else 'no dermatologist label'
    scores = {
        category: sum(weight for name, weight in weights.items()
                      if name in names and (name not in HEAD_ONLY or on_head))
        for category, names in SCIN_TO_CATEGORY.items()
    }
    best = max(scores, key=scores.get)
    if scores[best] < MIN_LABEL_WEIGHT:
        return None, 'not one of our categories'
    return best, ''

records, reasons = [], Counter()
for case_id, case in cases.items():
    label = labels_by_case.get(case_id, {})
    category, reason = categorise(case, label)
    if category is None or category not in CLASSES:
        reasons[reason or 'category not in CLASSES'] += 1
        continue
    for i in (1, 2, 3):
        path = case.get(f'image_{i}_path')
        if path:
            records.append({'url': SCIN + path, 'case_id': f'scin-{case_id}', 'category': category,
                            'source': 'SCIN', 'skin_tone': skin_tone(case, label),
                            'file': f'{category}/{case_id}_{i}.jpg'})

per_class_cases = defaultdict(set)
for r in records:
    per_class_cases[r['category']].add(r['case_id'])
print(f"\n{'category':<11} {'cases':>6} {'images':>7}")
for name in CLASSES:
    print(f"{name:<11} {len(per_class_cases[name]):>6} {sum(r['category'] == name for r in records):>7}")
print('\nleft out:', dict(reasons.most_common()))

In [ ]:
#@title 3b · Download the chosen SCIN photos (parallel, resized on the way in)
SCIN_DIR = DATA_ROOT / 'scin'

def fetch(record, attempts=3):
    target = SCIN_DIR / record['file']
    if target.exists():
        return record
    for attempt in range(attempts):
        try:
            response = requests.get(record['url'], timeout=60)
            response.raise_for_status()
            image = Image.open(io.BytesIO(response.content)).convert('RGB')
            image.thumbnail((MAX_IMAGE_SIDE, MAX_IMAGE_SIDE))
            target.parent.mkdir(parents=True, exist_ok=True)
            image.save(target, quality=92)
            return record
        except Exception:
            time.sleep(1 + attempt * 2)
    return None

started, done = time.time(), []
with ThreadPoolExecutor(max_workers=16) as pool:
    for n, result in enumerate(pool.map(fetch, records), 1):
        if result:
            done.append({**result, 'path': str(SCIN_DIR / result['file'])})
        if n % 100 == 0 or n == len(records):
            print(f'  {n}/{len(records)} downloaded ({time.time() - started:.0f}s)')
failed = len(records) - len(done)
print(f'{len(done)} SCIN photos ready' + (f', {failed} failed and were skipped' if failed else ''))
scin_records = done

In [ ]:
#@title 3c · Optional: extra acne / normal photos from huynhnhu213/acne (off by default)
from huggingface_hub import hf_hub_download

hf_records = []
if ADD_HF_ACNE_DATASET:
    # One zip: dataset-1/{train,validation,test}/{Acne,Non_Acne}/*.jpg. Read
    # directly -- load_dataset's folder inference mislabels this layout.
    archive = hf_hub_download('huynhnhu213/acne', 'dataset-1.zip', repo_type='dataset')
    rng = random.Random(SEED)
    with zipfile.ZipFile(archive) as zip_file:
        names = [n for n in zip_file.namelist() if n.lower().endswith('.jpg')]
        for folder, category in [('Acne', 'acne'), ('Non_Acne', 'normal')]:
            if category not in CLASSES:
                continue
            chosen = sorted(n for n in names if f'/train/{folder}/' in n)
            rng.shuffle(chosen)
            for name in chosen[:HF_ACNE_PER_CLASS]:
                target = DATA_ROOT / 'hf_acne' / category / Path(name).name
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(zip_file.read(name))
                hf_records.append({'path': str(target), 'case_id': f'hf-{target.stem}', 'category': category,
                                   'source': 'huynhnhu213/acne', 'skin_tone': 'unknown'})
    print(f'added {len(hf_records)} photos from huynhnhu213/acne')
else:
    print('skipped (ADD_HF_ACNE_DATASET = False)')

**The split is by case, not by photo.** A SCIN case can hold three photos of
the same skin. With one in training and another in validation, the score would
measure memory rather than skill — so every photo of a case goes to the same
side. The split is also stratified, so each category keeps its share.

In [ ]:
#@title 3d · Build the dataset and split train / validation by case
all_records = scin_records + hf_records

cases_per_class = defaultdict(set)
for r in all_records:
    cases_per_class[r['category']].add(r['case_id'])
LABELS = [c for c in CLASSES if len(cases_per_class[c]) >= MIN_CASES_PER_CLASS]
for c in CLASSES:
    if c not in LABELS:
        print(f'DROPPED {c!r}: only {len(cases_per_class[c])} cases (need {MIN_CASES_PER_CLASS}).')
assert len(LABELS) >= 2, 'Fewer than two categories have enough data.'
all_records = [r for r in all_records if r['category'] in LABELS]
id2label = {i: name for i, name in enumerate(LABELS)}
label2id = {name: i for i, name in enumerate(LABELS)}

case_category = {r['case_id']: r['category'] for r in all_records}
case_ids = sorted(case_category)
train_cases, val_cases = train_test_split(
    case_ids, test_size=VAL_FRACTION, random_state=SEED,
    stratify=[case_category[c] for c in case_ids],
)
train_cases = set(train_cases)
train_records = [r for r in all_records if r['case_id'] in train_cases]
val_records = [r for r in all_records if r['case_id'] not in train_cases]

FEATURES = Features({'image': ImageFeature(), 'label': ClassLabel(names=LABELS),
                     'case_id': Value('string'), 'source': Value('string'), 'skin_tone': Value('string')})

def to_dataset(rows):
    return Dataset.from_dict({
        'image': [r['path'] for r in rows],
        'label': [label2id[r['category']] for r in rows],
        'case_id': [r['case_id'] for r in rows],
        'source': [r['source'] for r in rows],
        'skin_tone': [r['skin_tone'] for r in rows],
    }, features=FEATURES)

train_ds, val_ds = to_dataset(train_records), to_dataset(val_records)
train_counts = np.bincount([label2id[r['category']] for r in train_records], minlength=len(LABELS))
val_counts = np.bincount([label2id[r['category']] for r in val_records], minlength=len(LABELS))

print(f'train {len(train_ds)} photos / validation {len(val_ds)} photos   classes: {LABELS}')
for i, name in enumerate(LABELS):
    note = '  <- small: treat its scores as rough' if train_counts[i] < 60 else ''
    print(f'  {name:<11} train {train_counts[i]:>5}   val {val_counts[i]:>4}{note}')
print('validation by skin tone:', dict(Counter(r['skin_tone'] for r in val_records)))

## 4 · Model selection

`MODEL_CHECKPOINT` (section 2) is loaded with its pretrained weights and a
**new classification head** sized to the categories that made it through 3d.
`ignore_mismatched_sizes` lets ResNet-50 drop its original 1,000-class head.

In [ ]:
#@title 4 · Load the pretrained model with a new head
model = AutoModelForImageClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)
total = sum(p.numel() for p in model.parameters())
print(f'{MODEL_CHECKPOINT}: {total / 1e6:.1f}M parameters, {len(LABELS)} outputs: {LABELS}')

## 5 · Preprocessing

`AutoImageProcessor` supplies what the backbone expects — input size and
normalisation. Training adds augmentation: photos arrive mirrored, tilted and
under every kind of light. Hue jitter is kept small on purpose — a large hue
shift would turn redness into something else.

In [ ]:
#@title 5a · Image processor and transforms
processor = AutoImageProcessor.from_pretrained(MODEL_CHECKPOINT)
size = processor.size
CROP = size.get('height') or size.get('shortest_edge') or 224
RESIZE = int(round(CROP * 256 / 224))
MEAN, STD = list(processor.image_mean), list(processor.image_std)
print(f'input {CROP}x{CROP}, resize shorter side to {RESIZE}, mean {MEAN}, std {STD}')

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(CROP, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.2, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
# Validation, the exported model and the app all use exactly this. It is
# written into labels.json in section 8.
eval_transform = transforms.Compose([
    transforms.Resize(RESIZE),
    transforms.CenterCrop(CROP),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

def with_transform(transform):
    def apply(batch):
        batch['pixel_values'] = [transform(image.convert('RGB')) for image in batch['image']]
        return batch
    return apply

train_ds.set_transform(with_transform(train_transform))
val_ds.set_transform(with_transform(eval_transform))

def collate(examples):
    return {
        'pixel_values': torch.stack([e['pixel_values'] for e in examples]),
        'labels': torch.tensor([e['label'] for e in examples]),
    }

In [ ]:
#@title 5b · Look at a few augmented training photos (do the labels make sense?)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, index in zip(axes.flat, random.sample(range(len(train_ds)), min(8, len(train_ds)))):
    example = train_ds[index]
    image = example['pixel_values'].permute(1, 2, 0).numpy() * np.array(STD) + np.array(MEAN)
    ax.imshow(np.clip(image, 0, 1)); ax.set_title(LABELS[example['label']]); ax.axis('off')
plt.tight_layout(); plt.show()

## 6 · Training

Hugging Face **Trainer** with mixed precision (`fp16`), a cosine schedule with
10% warmup, evaluation and a checkpoint **every epoch**, early stopping, and
the **best epoch by macro-F1** restored at the end.

**Class weights**: categories are unequal in size (dark spots especially), so
each category's loss is weighted by how rare it is — otherwise the model learns
that guessing the biggest category is usually right.

In [ ]:
#@title 6a · Metrics: accuracy and F1, computed at the end of every epoch
accuracy_metric = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')

def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=-1)
    references = eval_pred.label_ids
    return {
        'accuracy': accuracy_metric.compute(predictions=predictions, references=references)['accuracy'],
        # Macro: every category counts equally, so a small one failing shows.
        'f1_macro': f1_metric.compute(predictions=predictions, references=references, average='macro')['f1'],
        'f1_weighted': f1_metric.compute(predictions=predictions, references=references, average='weighted')['f1'],
    }

In [ ]:
#@title 6b · Train
CLASS_WEIGHTS = torch.tensor(
    np.clip(train_counts.sum() / (len(LABELS) * np.maximum(train_counts, 1)), 0.2, 10.0),
    dtype=torch.float32,
)
print('class weights:', {name: round(float(w), 2) for name, w in zip(LABELS, CLASS_WEIGHTS)})

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(
            outputs.logits, labels, weight=CLASS_WEIGHTS.to(outputs.logits.device))
        return (loss, outputs) if return_outputs else loss

steps_per_epoch = math.ceil(len(train_ds) / BATCH_SIZE)
training_args = TrainingArguments(
    output_dir='/content/checkpoints',
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    # 10% warmup, as a step count: works on transformers 4.x and 5.x alike.
    warmup_steps=max(1, int(0.1 * steps_per_epoch * EPOCHS)),
    lr_scheduler_type='cosine',
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_steps=max(1, steps_per_epoch // 2),
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    save_total_limit=2,
    dataloader_num_workers=2,
    remove_unused_columns=False,   # the 'image' column feeds the transforms
    report_to='none',
    seed=SEED,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collate,
    compute_metrics=compute_metrics,
    processing_class=processor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)

started = time.time()
train_result = trainer.train()
print(f'training took {(time.time() - started) / 60:.1f} min')

## 7 · Evaluation

Per-epoch accuracy and F1; then the restored **best** model in detail —
per-category report, confusion matrix, and accuracy **by skin tone** (SCIN's
Fitzpatrick labels; NFR-SAFE-008 requires this breakdown) and **by source**.

If the optional acne dataset is on and SCIN-only accuracy is much lower than
overall, the model is leaning on photo style rather than skin.

In [ ]:
#@title 7a · Accuracy and F1 per epoch
history = [h for h in trainer.state.log_history if 'eval_f1_macro' in h]
print(f"{'epoch':>5}  {'accuracy':>8}  {'F1 macro':>8}  {'F1 weighted':>11}  {'val loss':>8}")
for h in history:
    print(f"{h['epoch']:>5.0f}  {h['eval_accuracy']:>8.3f}  {h['eval_f1_macro']:>8.3f}  "
          f"{h['eval_f1_weighted']:>11.3f}  {h['eval_loss']:>8.3f}")
best = max(history, key=lambda h: h['eval_f1_macro'])
print(f"\nbest epoch {best['epoch']:.0f}: accuracy {best['eval_accuracy']:.3f}, F1 macro {best['eval_f1_macro']:.3f}")

In [ ]:
#@title 7b · Best model: per-category report and confusion matrix
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt

final_metrics = trainer.evaluate()
prediction = trainer.predict(val_ds)
predicted = np.argmax(prediction.predictions, axis=-1)
actual = prediction.label_ids

print(classification_report(actual, predicted, labels=list(range(len(LABELS))), target_names=LABELS,
                            digits=3, zero_division=0))
PER_CLASS = classification_report(actual, predicted, labels=list(range(len(LABELS))), target_names=LABELS,
                                  output_dict=True, zero_division=0)

fig, ax = plt.subplots(figsize=(1.3 * len(LABELS) + 3, 1.3 * len(LABELS) + 2))
ConfusionMatrixDisplay.from_predictions(actual, predicted, labels=list(range(len(LABELS))),
                                        display_labels=LABELS, cmap='Purples', xticks_rotation=45,
                                        ax=ax, colorbar=False)
ax.set_title('Validation confusion matrix (rows = true category)')
plt.tight_layout(); plt.show()

In [ ]:
#@title 7c · Accuracy by skin tone and by source
def breakdown(key):
    groups = defaultdict(list)
    for record, truth, guess in zip(val_records, actual, predicted):
        groups[record[key]].append(truth == guess)
    return {group: (float(np.mean(hits)), len(hits)) for group, hits in sorted(groups.items())}

BY_TONE, BY_SOURCE = breakdown('skin_tone'), breakdown('source')
for title, table in [('skin tone (Fitzpatrick)', BY_TONE), ('source', BY_SOURCE)]:
    print(f'accuracy by {title}:')
    for group, (accuracy, n) in table.items():
        note = '  (fewer than 20 photos: rough)' if n < 20 else ''
        print(f'  {group:<18} {accuracy:6.1%}  of {n}{note}')
known = [v for k, v in BY_TONE.items() if k != 'unknown' and v[1] >= 20]
if len(known) >= 2:
    print(f'\nlargest gap between skin-tone groups: {max(a for a, _ in known) - min(a for a, _ in known):.1%}')

## 8 · Export

Written to `OUTPUT_DIR`, then zipped and **downloaded to your computer**
(and copied to Drive if `SAVE_TO_DRIVE` is on):

| File | For |
|---|---|
| `hf_model/` | fine-tuned model + processor — reload with `from_pretrained`, or train further |
| `skin_disease_model.onnx` | ONNX Runtime: Python/Node backend, React Native (`onnxruntime-react-native`), Flutter |
| `skin_disease_model.int8.onnx` | the same, ~4× smaller (8b) |
| `labels.json` | category order, preprocessing, metrics, data sources — what the app needs to use the ONNX file |
| `NOTICE.txt` | the SCIN attribution its licence requires when you share the model |

The ONNX model takes `pixel_values` (1×3×H×W, preprocessed as in 5a) and returns
`probabilities` with softmax applied. Everything a consumer needs to reproduce
that preprocessing is in `labels.json`; 8c shows it end to end without PyTorch.

In [ ]:
#@title 8a · Save the model and export ONNX
import onnxruntime as ort
from torch.utils.data import DataLoader

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HF_DIR = OUTPUT_DIR / 'hf_model'
trainer.save_model(str(HF_DIR))          # the best epoch, restored by the Trainer
processor.save_pretrained(str(HF_DIR))

# Reload on the CPU with plain ("eager") attention: the fused attention used in
# training does not always export cleanly.
try:
    export_model = AutoModelForImageClassification.from_pretrained(str(HF_DIR), attn_implementation='eager')
except (TypeError, ValueError):
    export_model = AutoModelForImageClassification.from_pretrained(str(HF_DIR))
export_model.eval()

class WithSoftmax(torch.nn.Module):
    # Probabilities out, so the app needs no post-processing of its own.
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, pixel_values):
        return torch.softmax(self.model(pixel_values=pixel_values).logits, dim=-1)

wrapped = WithSoftmax(export_model).eval()
onnx_path = OUTPUT_DIR / 'skin_disease_model.onnx'
export_kwargs = dict(
    input_names=['pixel_values'], output_names=['probabilities'],
    dynamic_axes={'pixel_values': {0: 'batch'}, 'probabilities': {0: 'batch'}},
    opset_version=17,
)
dummy = torch.randn(1, 3, CROP, CROP)
with torch.no_grad():
    try:
        torch.onnx.export(wrapped, (dummy,), str(onnx_path), dynamo=False, **export_kwargs)
    except TypeError:   # older torch without the `dynamo` switch
        torch.onnx.export(wrapped, (dummy,), str(onnx_path), **export_kwargs)

# Check the ONNX file against PyTorch on real validation photos instead of
# trusting the exporter (the ml/README gate requires a match).
batch = next(iter(DataLoader(val_ds, batch_size=8, collate_fn=collate)))['pixel_values']
with torch.no_grad():
    expected = wrapped(batch).numpy()
session = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
got = session.run(None, {'pixel_values': batch.numpy()})[0]
ONNX_DIFF = float(np.abs(expected - got).max())
print(f'ONNX vs PyTorch: max difference {ONNX_DIFF:.2e}, '
      f'same top prediction {(expected.argmax(1) == got.argmax(1)).mean():.0%}')
assert ONNX_DIFF < 1e-4, 'ONNX output does not match PyTorch -- do not use this file.'

# Backend concern ids. On screen the app names concerns only through the
# server's concernLabels (IF-UI-001), never these.
CONCERN_IDS = {'acne': 'ACNE', 'dark_spots': 'UNEVEN_TONE', 'redness': 'MILD_REDNESS'}
labels_json = {
    'model': RUN_NAME,
    'base_model': MODEL_CHECKPOINT,
    'trained_at': time.strftime('%Y-%m-%dT%H:%M:%S'),
    'classes': LABELS,
    'concern_ids': {name: CONCERN_IDS.get(name) for name in LABELS},
    'multi_label': False,
    'threshold': 0.5,
    'input': {'name': 'pixel_values', 'shape': [None, 3, CROP, CROP], 'layout': 'NCHW', 'color': 'RGB',
              'scale': '0-1', 'mean': MEAN, 'std': STD, 'resize_shorter_side': RESIZE, 'center_crop': CROP},
    'output': {'name': 'probabilities', 'activation': 'softmax'},
    'images': {'train': len(train_records), 'validation': len(val_records),
               'train_per_class': {n: int(c) for n, c in zip(LABELS, train_counts)},
               'validation_per_class': {n: int(c) for n, c in zip(LABELS, val_counts)}},
    'validation': {k.replace('eval_', ''): round(float(v), 4) for k, v in final_metrics.items()
                   if k in ('eval_accuracy', 'eval_f1_macro', 'eval_f1_weighted', 'eval_loss')},
    'per_class': {n: {k: round(float(v), 4) for k, v in PER_CLASS[n].items()} for n in LABELS},
    'accuracy_by_skin_tone': {k: {'accuracy': round(a, 4), 'n': n} for k, (a, n) in BY_TONE.items()},
    'accuracy_by_source': {k: {'accuracy': round(a, 4), 'n': n} for k, (a, n) in BY_SOURCE.items()},
    'data_sources': {
        'SCIN': {'url': 'https://github.com/google-research-datasets/scin', 'license': 'SCIN Data Use License',
                 'license_url': SCIN_LICENSE_URL, 'mapping': SCIN_TO_CATEGORY,
                 'min_label_weight': MIN_LABEL_WEIGHT, 'face_only': FACE_ONLY},
        **({'huynhnhu213/acne': {'url': 'https://huggingface.co/datasets/huynhnhu213/acne',
                                 'license': 'apache-2.0 (declared; image provenance not stated)',
                                 'per_class': HF_ACNE_PER_CLASS}} if ADD_HF_ACNE_DATASET else {}),
    },
    'placeholder_classes': [],
    'export_check': {'onnx_max_abs_diff': ONNX_DIFF},
    'deployable': False,
    'deployable_reason': 'Not yet evaluated against the FR-AI-010 gate (ml/README.md).',
}
(OUTPUT_DIR / 'labels.json').write_text(json.dumps(labels_json, indent=2))

# SCIN licence, Section 3(a): credit the source, link the licence, and say it
# was modified, whenever the material or something adapted from it is shared.
(OUTPUT_DIR / 'NOTICE.txt').write_text(
    'This model was trained on images and labels from the Skin Condition Image Network (SCIN)\n'
    'dataset by Google, used under the SCIN Data Use License:\n'
    f'  {SCIN_LICENSE_URL}\n'
    'Images were selected by label, resized and augmented for training. Re-identifying any\n'
    'person pictured in SCIN is prohibited by that licence.\n'
)
print('saved to', OUTPUT_DIR)

In [ ]:
#@title 8b · A ~4x smaller ONNX for phones (int8)
QUANTIZE = True  #@param {type:"boolean"}

if QUANTIZE:
    from onnxruntime.quantization import QuantType, quantize_dynamic

    int8_path = OUTPUT_DIR / 'skin_disease_model.int8.onnx'
    quantize_dynamic(str(onnx_path), str(int8_path), weight_type=QuantType.QInt8)

    # Smaller only helps if it still agrees with the full model.
    full = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
    small = ort.InferenceSession(str(int8_path), providers=['CPUExecutionProvider'])
    agree = correct_full = correct_small = seen = 0
    for batch in DataLoader(val_ds, batch_size=32, collate_fn=collate):
        x, y = batch['pixel_values'].numpy(), batch['labels'].numpy()
        p_full = full.run(None, {'pixel_values': x})[0].argmax(1)
        p_small = small.run(None, {'pixel_values': x})[0].argmax(1)
        agree += (p_full == p_small).sum(); correct_full += (p_full == y).sum()
        correct_small += (p_small == y).sum(); seen += len(y)
    print(f'size: {onnx_path.stat().st_size / 1e6:.0f} MB -> {int8_path.stat().st_size / 1e6:.0f} MB')
    print(f'accuracy: full {correct_full / seen:.3f}, int8 {correct_small / seen:.3f}, '
          f'same prediction {agree / seen:.1%}')

    # Recorded, because the metrics in labels.json were measured on the full
    # model. The backend serves the int8 file only if this agreement meets its
    # gate (FR-AI-009: the evaluated artefact must be the served one).
    labels_json['int8_check'] = {
        'file': int8_path.name, 'n': int(seen), 'agreement': round(float(agree / seen), 4),
        'accuracy_full': round(float(correct_full / seen), 4),
        'accuracy_int8': round(float(correct_small / seen), 4),
    }
    (OUTPUT_DIR / 'labels.json').write_text(json.dumps(labels_json, indent=2))

In [ ]:
#@title 8c · Try the exported ONNX model on a few validation photos
# Uses only numpy, Pillow and ONNX Runtime -- no PyTorch -- and reads every
# preprocessing setting from labels.json. This is the code a Python backend
# needs; a phone app does the same steps in its own language.
spec = json.loads((OUTPUT_DIR / 'labels.json').read_text())
session = ort.InferenceSession(str(OUTPUT_DIR / 'skin_disease_model.onnx'), providers=['CPUExecutionProvider'])

def preprocess(path, spec=spec['input']):
    image = Image.open(path).convert('RGB')
    scale = spec['resize_shorter_side'] / min(image.size)
    image = image.resize((max(spec['resize_shorter_side'], round(image.width * scale)),
                          max(spec['resize_shorter_side'], round(image.height * scale))), Image.Resampling.BILINEAR)
    crop = spec['center_crop']
    left, top = (image.width - crop) // 2, (image.height - crop) // 2
    image = image.crop((left, top, left + crop, top + crop))
    array = (np.asarray(image, dtype=np.float32) / 255.0 - np.array(spec['mean'], np.float32)) / np.array(spec['std'], np.float32)
    return array.transpose(2, 0, 1)[None]  # HWC -> NCHW

for record in random.Random(SEED).sample(val_records, min(4, len(val_records))):
    probabilities = session.run(None, {spec['input']['name']: preprocess(record['path'])})[0][0]
    ranked = sorted(zip(spec['classes'], probabilities), key=lambda item: -item[1])
    print(f"true: {record['category']:<11} predicted: " + ', '.join(f'{name} {p:.0%}' for name, p in ranked))

In [ ]:
#@title 8d · Download the model (and optionally copy it to Drive)
for path in sorted(OUTPUT_DIR.rglob('*')):
    if path.is_file():
        print(f'{path.stat().st_size / 1e6:9.1f} MB  {path.relative_to(OUTPUT_DIR)}')

archive = shutil.make_archive(str(OUTPUT_DIR), 'zip', root_dir=OUTPUT_DIR)
print('\nzipped:', archive, f'({Path(archive).stat().st_size / 1e6:.0f} MB)')

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_copy = Path('/content/drive/MyDrive/skinsight/models') / Path(archive).name
    drive_copy.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(archive, drive_copy)
    print('copied to Drive:', drive_copy)

from google.colab import files
files.download(archive)

## Next steps

* **Using it outside Colab:** the zip holds `skin_disease_model.onnx` and
  `labels.json`; the `preprocess` function in 8c is the reference for feeding
  it a photo.
* **Read 7b and 7c before trusting it.** A category with few validation photos
  (dark spots especially) has a wide margin of error, and the skin-tone table
  shows whether it works for everyone — not just the majority.
* **Before any user sees it:** the FR-AI-010 gate in `ml/README.md`, and an
  Appendix I register entry with the sources and licences recorded in
  `labels.json`.
* **Sharing the model or its outputs:** keep `NOTICE.txt` with it — the SCIN
  licence requires the attribution.